In [3]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

from sklearn.linear_model import LinearRegression

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)

from xgboost import XGBRegressor


In [4]:

def prepare_reliability_features(df):

    df = df.copy()

    temperature_kelvin = (
        df["Stress_Temperature_C"]
        + 273.15
    )

    df["Inverse_Temperature"] = (
        1 / temperature_kelvin
    )

    df["Log_Voltage"] = np.log(
        df["Stress_Voltage_V"].clip(lower=1.0)
    )

    df["Humidity_Fraction"] = (
        df["Humidity_Percent"] / 100
    )

    return df


def create_training_dataset(df):

    df = prepare_reliability_features(df)

    failure_data = df[
        df["Censored"] == 0
    ].copy()

    X = failure_data[
        [
            "Inverse_Temperature",
            "Log_Voltage",
            "Humidity_Fraction"
        ]
    ]

    y = np.log(
        failure_data[
            "Failure_Time_Hours"
        ]
    )

    return train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )


def evaluate_model(
    model,
    model_name,
    X_test,
    y_test
):

    predictions = model.predict(
        X_test
    )

    r2 = r2_score(
        y_test,
        predictions
    )

    mae = mean_absolute_error(
        y_test,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            predictions
        )
    )

    return {
        "Model": model_name,
        "R2": r2,
        "MAE": mae,
        "RMSE": rmse
    }


def train_all_models(df):

    (
        X_train,
        X_test,
        y_train,
        y_test
    ) = create_training_dataset(df)

    models = {

        "Physics_Linear_Model":
        LinearRegression(),

        "Random_Forest":
        RandomForestRegressor(
            n_estimators=300,
            max_depth=8,
            random_state=42
        ),

        "Gradient_Boosting":
        GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            random_state=42
        ),

        "XGBoost":
        XGBRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=42
        )
    }

    results = []

    trained_models = {}

    for name, model in models.items():

        model.fit(
            X_train,
            y_train
        )

        metrics = evaluate_model(
            model,
            name,
            X_test,
            y_test
        )

        results.append(
            metrics
        )

        trained_models[name] = model

    results_df = pd.DataFrame(
        results
    )

    results_df = (
        results_df
        .sort_values(
            by="R2",
            ascending=False
        )
    )

    print(
        "\nMODEL PERFORMANCE COMPARISON"
    )

    print(
        "-" * 60
    )

    print(
        results_df.to_string(
            index=False
        )
    )

    best_model_name = (
        results_df.iloc[0]["Model"]
    )

    print(
        f"\nBEST MODEL : "
        f"{best_model_name}"
    )

    return (
        trained_models,
        results_df
    )


def predict_device_lifetime(
    model,
    temperature_c,
    voltage_v,
    humidity_percent
):

    inverse_temperature = (
        1 /
        (
            temperature_c
            + 273.15
        )
    )

    log_voltage = np.log(
        max(voltage_v, 1.0)
    )

    humidity_fraction = (
        humidity_percent / 100
    )

    X_new = pd.DataFrame(
        {
            "Inverse_Temperature":
            [inverse_temperature],

            "Log_Voltage":
            [log_voltage],

            "Humidity_Fraction":
            [humidity_fraction]
        }
    )

    predicted_log_life = (
        model.predict(
            X_new
        )[0]
    )

    predicted_lifetime = np.exp(
        predicted_log_life
    )

    return predicted_lifetime


def run_ai_reliability_engine():

    df = pd.read_csv(
        "reliability_dataset1.csv"
    )

    (
        trained_models,
        results_df
    ) = train_all_models(df)

    best_model_name = (
        results_df.iloc[0]["Model"]
    )

    best_model = (
        trained_models[
            best_model_name
        ]
    )

    predicted_lifetime = (
        predict_device_lifetime(
            best_model,
            temperature_c=110,
            voltage_v=4.5,
            humidity_percent=65
        )
    )

    print(
        "\nAI RELIABILITY PREDICTION"
    )

    print(
        "-" * 60
    )

    print(
        f"Selected Model : "
        f"{best_model_name}"
    )

    print(
        f"Predicted Lifetime : "
        f"{predicted_lifetime:.2f}"
        f" Hours"
    )


run_ai_reliability_engine()


MODEL PERFORMANCE COMPARISON
------------------------------------------------------------
               Model       R2      MAE     RMSE
       Random_Forest 0.449505 0.434360 0.564406
             XGBoost 0.449420 0.434442 0.564449
   Gradient_Boosting 0.449337 0.434543 0.564492
Physics_Linear_Model 0.055483 0.591542 0.739298

BEST MODEL : Random_Forest

AI RELIABILITY PREDICTION
------------------------------------------------------------
Selected Model : Random_Forest
Predicted Lifetime : 1110.69 Hours
